# 07 · The world model as a plan critic: pick which demo to execute, before acting

In each of LIBERO's 50 evaluation scenes the robot has 13 candidate plans: the object-relative retargeting of each of the 13 human demos, each with a random perturbation of its grasp and place point (as in data generation). Some candidates work in this scene, some do not (the scripted teacher succeeds ~80% of the time).

**World-model choice:** turn every candidate into its action sequence with a nominal model of the arm (no simulator: it knows the arm, not the bowl), let the world model imagine each sequence from the scene's real first frame, and execute the candidate whose imagined bowl ends closest to the plate. **Compared with** a random candidate (the mean success of the 13), always the same demo chosen in advance (`65ee`, the highest generation yield), and the oracle (some candidate works). The simulator runs every candidate only to score these rules. Code and details: `p2p/wm/plan_select.py`.

**Runtime:** T4 GPU (renders one frame per scene, runs the world model); the ~650 physics rollouts for scoring run on the CPU workers. About 35-50 min in total.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"], check=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"
INITS, PER_DEMO = 50, 1        # 50 scenes x 13 demos x 1 perturbation = 650 candidates
WORKERS = os.cpu_count()
OUT = f"{BASE}/plan_select"
from p2p.hub import pull, push
pull(HUB_REPO, f"{BASE}/hub", allow_patterns=["wm/run/wm.pt"])

Optional check (~2 min): refit the nominal arm model and compare its waypoint timings with the simulator's (should be within a few steps).

In [ ]:
# !python tools/fit_arm_model.py

In [ ]:
!python -m p2p.wm.plan_select {BASE}/hub/wm/run/wm.pt {OUT} --inits {INITS} --per-demo {PER_DEMO} --workers {WORKERS}

In [ ]:
import pandas as pd
summary = json.load(open(f"{OUT}/plan_select_summary.json"))
print(json.dumps({k: v for k, v in summary.items() if k != "per_scene"}, indent=1))
c = pd.read_csv(f"{OUT}/plan_select_candidates.csv")
chosen = c.loc[c.groupby("init")["wm_dist_nominal_cm"].idxmin()]
per_demo = c.groupby("demo").agg(success=("success", "mean"), imagined_cm=("wm_dist_nominal_cm", "median"))
per_demo["times_chosen"] = chosen["demo"].value_counts()
display(per_demo.fillna(0).sort_values("success", ascending=False).round(2))

In [ ]:
push(OUT, HUB_REPO, path_in_repo="plan_select")
!cd {BASE} && zip -qr plan_select.zip plan_select
try:
    from google.colab import files; files.download(f"{BASE}/plan_select.zip")
except Exception:
    print("download", f"{BASE}/plan_select.zip", "from the Output panel")